# Hetionet 서브그래프 구축

Hetionet v1.0에서 Gene / Pathway / Disease / Biological Process 노드와
관련 엣지만 추려 NetworkX MultiDiGraph를 만든다.

In [ ]:
# 데이터 다운로드
#
# 주의할 점 2가지
# 1) 저장소가 dhimmel/hetionet -> hetio/hetionet 으로 이름이 바뀌었다.
#    raw.githubusercontent.com은 옛 이름도 리다이렉트해 주지만,
#    LFS 파일을 주는 media.githubusercontent.com은 404를 낸다. -> hetio 를 사용.
# 2) 파일명에 버전이 붙는다. nodes.tsv (X) -> hetionet-v1.0-nodes.tsv (O)

import gzip
import shutil
import urllib.request
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)   # urlretrieve는 폴더를 만들어주지 않는다

BASE     = "https://raw.githubusercontent.com/hetio/hetionet/main/hetnet/tsv"
LFS_BASE = "https://media.githubusercontent.com/media/hetio/hetionet/main/hetnet/tsv"

nodes_path    = DATA_DIR / "nodes.tsv"
edges_gz_path = DATA_DIR / "edges.sif.gz"
edges_path    = DATA_DIR / "edges.sif"

if not nodes_path.exists():
    urllib.request.urlretrieve(f"{BASE}/hetionet-v1.0-nodes.tsv", nodes_path)

if not edges_gz_path.exists():
    # edges는 Git LFS로 관리되므로 raw URL은 포인터 텍스트(133 B)만 준다. media URL 필수.
    urllib.request.urlretrieve(f"{LFS_BASE}/hetionet-v1.0-edges.sif.gz", edges_gz_path)

if not edges_path.exists():
    with gzip.open(edges_gz_path, "rb") as f_in, open(edges_path, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

print("nodes.tsv :", f"{nodes_path.stat().st_size:,} bytes")
print("edges.sif :", f"{edges_path.stat().st_size:,} bytes")

In [ ]:
# 데이터 로드 (기존 코드에서 nodes / edges 를 만드는 셀이 빠져 NameError가 났던 부분)

import pandas as pd
import networkx as nx

nodes = pd.read_csv(nodes_path, sep="\t")   # 컬럼: id, name, kind
edges = pd.read_csv(edges_path, sep="\t")   # 컬럼: source, metaedge, target

print(nodes.shape, edges.shape)
display(nodes.head(3))
display(edges.head(3))

In [ ]:
KEEP_NODE_TYPES = {"Gene", "Pathway", "Disease", "Biological Process"}
KEEP_EDGE_TYPES = {"GpPW", "GiG", "Gr>G", "DaG", "DuG", "DdG", "GpBP"}

# 필요한 노드만 추려 ID 세트와 속성 조회용 dict를 한 번에 만든다
keep_nodes    = nodes[nodes["kind"].isin(KEEP_NODE_TYPES)]
keep_node_ids = set(keep_nodes["id"])
node_info     = keep_nodes.set_index("id")[["name", "kind"]].to_dict("index")

# pandas 벡터화 필터링 (핵심 성능 최적화)
mask = (
    edges["metaedge"].isin(KEEP_EDGE_TYPES) &
    edges["source"].isin(keep_node_ids) &
    edges["target"].isin(keep_node_ids)
)
filtered_edges = edges[mask]

print(f"kept nodes     : {len(keep_node_ids):,}")
print(f"filtered edges : {len(filtered_edges):,}")

In [ ]:
# NetworkX 그래프 구축 (MultiDiGraph: 방향 + 다중 엣지)
G_sub = nx.MultiDiGraph()

# 노드 추가
for nid in keep_node_ids:
    info = node_info[nid]
    G_sub.add_node(nid, name=info["name"], kind=info["kind"])

# 엣지 추가
for row in filtered_edges.itertuples(index=False):
    G_sub.add_edge(row.source, row.target, metaedge=row.metaedge)

# 고립 노드 제거
isolates = list(nx.isolates(G_sub))
G_sub.remove_nodes_from(isolates)

print(f"G_sub: {G_sub.number_of_nodes():,} nodes, {G_sub.number_of_edges():,} edges")
print(f"removed isolates: {len(isolates):,}")

In [ ]:
# 노드 종류별 요약
from collections import Counter

kind_counts = Counter(kind for _, kind in G_sub.nodes(data="kind"))
for kind, n in sorted(kind_counts.items(), key=lambda kv: -kv[1]):
    print(f"{kind:20s} {n:>7,}")